# 10 — Intro to Knowledge Graphs

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Understand nodes, edges, relationships.
2. See why graphs are useful for *connection* questions ("who approved what?").
3. Build a small graph from our synthetic data using NetworkX.
4. Run three useful graph queries.


## A knowledge graph in one minute

* A **node** is a thing (a vendor, an employee, an invoice).
* An **edge** is a relationship (`issued`, `approved_by`, `related_to`).
* A **graph** is just lots of nodes and edges together.

Where text-based RAG is good at *"what does the policy say about X?"*, graphs are good at *"who approved which payment to which related party?"* — relationship-heavy questions where the answer is a path, not a paragraph.

In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 10.1 — Build the graph

In [ ]:
from src.graph_utils import build_finance_graph, graph_summary
G = build_finance_graph()
summary = graph_summary(G)
print('Total nodes:', summary['nodes'])
print('Total edges:', summary['edges'])
print('Nodes by type:')
for k, v in summary['node_types'].items():
    print(f'  {k:12s} {v}')
print('Edges by relation:')
for k, v in summary['edge_types'].items():
    print(f'  {k:18s} {v}')

## 10.2 — Visualise a small subgraph

In [ ]:
import matplotlib.pyplot as plt
from src.graph_utils import draw_subgraph

# Pick: the company, two related-party vendors, their first 2 invoices each, and their approvers
nodes = ['Himal Trading', 'V004', 'V010', 'E001', 'E002', 'E003']
for n in list(nodes):
    if n.startswith('V'):
        out = [b for _, b, d in G.out_edges(n, data=True) if d.get('rel') == 'issued'][:2]
        nodes.extend(out)
fig, ax = plt.subplots(figsize=(10, 7))
draw_subgraph(G, nodes, ax=ax, title='Related-party vendors and approvers (sample)')
plt.show()

## 10.3 — Query 1: all invoices issued by a specific vendor

In [ ]:
from src.graph_utils import find_invoices_for_vendor
for inv in find_invoices_for_vendor(G, 'V004')[:10]:
    print(f"  {inv['invoice']:18s} NPR {inv['amount']:>12,.0f}  (date {inv.get('date')})")

## 10.4 — Query 2: employees who approved high-value invoices

In [ ]:
from src.graph_utils import find_high_value_approvers
for row in find_high_value_approvers(G, threshold=500_000)[:15]:
    print(f"  {row['invoice']:18s} NPR {row['amount']:>12,.0f}  approved by  {row['approver']}  ({row['approver_name']})")

## 10.5 — Query 3: related-party transaction paths

In [ ]:
from src.graph_utils import find_related_party_paths
paths = find_related_party_paths(G)
print(f'{len(paths)} related-party invoice paths:')
for p in paths[:15]:
    print(f"  {p['vendor_name']:35s} → {p['invoice']:16s} NPR {p['amount']:>10,.0f} → approved by {p['approver_name']}")

## Expected output

* ~120 invoice nodes, ~80 JE nodes, 12 vendors, 8 employees.
* The plot shows two related-party vendors (V004 Annapurna, V010 Himal Family) with sample invoices and their approvers.
* Query 3 lists every invoice issued by a related-party vendor and who approved it.


## Exercise

1. Add an edge to mark `E001` (CEO) as the *spouse* of someone at vendor `V010`. (Hint: `G.add_edge('E001', 'V010', rel='family_of')`).
2. Write a query that returns every employee with a `family_of` edge to a vendor that issued an invoice.
3. Sketch on paper: what other relationships from your firm's day-to-day work would be valuable as a graph?

## Common errors

| Symptom | Fix |
|---|---|
| `KeyError` on a node | The node wasn't added — check the vendor / employee code. |
| Visualisation is unreadable | Pass a smaller `nodes` list; spring layout works best for <30 nodes. |


## ⚠️ Professional caution

Graphs make **connections** very visible — including connections the data does not actually support. *Always* verify that the relationship encoded as an edge is supported by source evidence.